# SCEDG: Wiki6M LongCLIP-L index

Run this notebook **once** before [`Pipeline.ipynb`](Pipeline.ipynb).

It streams `Wiki6M_ver_1_0.jsonl.gz`, the list of about six million Wikipedia entities released with the OVEN benchmark, encodes each entity as `"<title> is <first 300 characters of the Wikipedia summary>"` with LongCLIP-L, and writes FAISS `IndexFlatIP` chunks plus matching metadata to `<ROOT_DIR>/KB/Chunk_files/Long-CLIP-L_Wiki6M`.

A small progress file (`index_progress.json`) is written after each completed chunk, so an interrupted run resumes without rebuilding finished chunks.

Set `ROOT_DIR` to the same directory used by `Pipeline.ipynb`. If the Wiki6M file is not in `<ROOT_DIR>/KB/`, the download cell fetches it from the official OVEN storage.

# 1. Environment

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
!pip install -q faiss-cpu ftfy regex tqdm torch torchvision


In [ ]:
import gc
import gzip
import json
import os
import subprocess
import sys

import faiss
import numpy as np
import torch
from tqdm.auto import tqdm

if not os.path.exists("Long-CLIP"):
    subprocess.run(
        ["git", "clone", "https://github.com/beichenzbc/Long-CLIP.git"],
        check=True,
    )

longclip_path = os.path.abspath("Long-CLIP")
if longclip_path not in sys.path:
    sys.path.append(longclip_path)

from model import longclip


# 2. Configuration

In [ ]:
ROOT_DIR = "/content/drive/MyDrive/dataset"
WIKI6M_PATH = os.path.join(ROOT_DIR, "KB", "Wiki6M_ver_1_0.jsonl.gz")
CHUNKS_DIR = os.path.join(
    ROOT_DIR, "KB", "Chunk_files", "Long-CLIP-L_Wiki6M"
)

LONGCLIP_CHECKPOINT = "longclip-L.pt"
LONGCLIP_URL = (
    "https://huggingface.co/BeichenZhang/LongCLIP-L/"
    "resolve/main/longclip-L.pt"
)
BATCH_SIZE = 1024
CHUNK_SIZE = 512000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROGRESS_FILE = os.path.join(CHUNKS_DIR, "index_progress.json")

os.makedirs(CHUNKS_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Wiki6M: {WIKI6M_PATH}")
print(f"Output: {CHUNKS_DIR}")


## 2.1. Download Wiki6M if missing

In [ ]:
WIKI6M_URL = (
    "http://storage.googleapis.com/gresearch/open-vision-language/"
    "Wiki6M_ver_1_0.jsonl.gz"
)

if not os.path.exists(WIKI6M_PATH):
    os.makedirs(os.path.dirname(WIKI6M_PATH), exist_ok=True)
    partial_path = WIKI6M_PATH + ".part"
    subprocess.run(["wget", "-q", "-O", partial_path, WIKI6M_URL], check=True)
    os.replace(partial_path, WIKI6M_PATH)

print(f"Wiki6M: {WIKI6M_PATH} ({os.path.getsize(WIKI6M_PATH) / 1e9:.2f} GB)")

# 3. Build the index

In [ ]:
class LongCLIPTextEncoder:
    def __init__(self):
        if not os.path.exists(LONGCLIP_CHECKPOINT):
            subprocess.run(
                ["wget", "-q", LONGCLIP_URL, "-O", LONGCLIP_CHECKPOINT],
                check=True,
            )
        self.model, _ = longclip.load(LONGCLIP_CHECKPOINT, device=DEVICE)
        self.model.eval()
        self.tokenize = longclip.tokenize
        with torch.inference_mode():
            probe = self.model.encode_text(
                self.tokenize(["dimension probe"], truncate=True).to(DEVICE)
            )
        self.dimension = int(probe.shape[-1])

    def encode(self, texts):
        with torch.inference_mode():
            tokens = self.tokenize(texts, truncate=True).to(DEVICE)
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)
        return features.float().cpu().numpy()


def wiki6m_batches(path, batch_size, skip_lines=0):
    texts, metadata = [], []
    line_number = 0
    with gzip.open(path, "rt", encoding="utf-8") as file:
        for line in file:
            line_number += 1
            if line_number <= skip_lines:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            qid = item.get("wikidata_id")
            title = item.get("wikipedia_title")
            description = (
                item.get("wikipedia_summary")
                or item.get("wikipedia_content")
                or ""
            )
            if not qid or not title or not description:
                continue

            short_description = description[:300]
            texts.append(f"{title} is {short_description}")
            metadata.append(
                {
                    "id": qid,
                    "label": title,
                    "desc": short_description,
                }
            )
            if len(texts) == batch_size:
                yield texts, metadata, line_number
                texts, metadata = [], []

    if texts:
        yield texts, metadata, line_number


def load_progress():
    if not os.path.exists(PROGRESS_FILE):
        existing = [
            name for name in os.listdir(CHUNKS_DIR)
            if name.endswith(".index")
        ]
        if existing:
            raise RuntimeError(
                "Existing index chunks were found without index_progress.json. "
                "Move or remove them before starting a clean build."
            )
        return {"source_lines_consumed": 0, "next_chunk_id": 0}
    with open(PROGRESS_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def save_chunk(index, metadata, chunk_id, source_lines_consumed):
    index_path = os.path.join(
        CHUNKS_DIR, f"wikidata_text_longclip_part{chunk_id}.index"
    )
    metadata_path = os.path.join(
        CHUNKS_DIR, f"metadata_text_longclip_part{chunk_id}.json"
    )
    faiss.write_index(index, index_path)
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False)
    with open(PROGRESS_FILE, "w", encoding="utf-8") as file:
        json.dump(
            {
                "source_lines_consumed": source_lines_consumed,
                "next_chunk_id": chunk_id + 1,
            },
            file,
        )


def build_wiki6m_index():
    if not os.path.exists(WIKI6M_PATH):
        raise FileNotFoundError(WIKI6M_PATH)

    progress = load_progress()
    encoder = LongCLIPTextEncoder()
    chunk_id = int(progress["next_chunk_id"])
    skip_lines = int(progress["source_lines_consumed"])

    index = faiss.IndexFlatIP(encoder.dimension)
    metadata_buffer = []
    last_line = skip_lines

    for texts, metadata, line_number in tqdm(
        wiki6m_batches(WIKI6M_PATH, BATCH_SIZE, skip_lines),
        desc="Encoding Wiki6M",
    ):
        vectors = encoder.encode(texts).astype("float32")
        index.add(vectors)
        metadata_buffer.extend(metadata)
        last_line = line_number

        if index.ntotal >= CHUNK_SIZE:
            save_chunk(index, metadata_buffer, chunk_id, last_line)
            chunk_id += 1
            index = faiss.IndexFlatIP(encoder.dimension)
            metadata_buffer = []
            gc.collect()

    if index.ntotal:
        save_chunk(index, metadata_buffer, chunk_id, last_line)

    print("Wiki6M indexing complete.")


build_wiki6m_index()
